# 칼로리 소모량 예측 AI (Original Version)

이 노트북은 대회 당시 사용했던 원본 실험 과정을 정리한 것입니다.

## 📌 목표
운동 데이터(심박수, 시간, 체온 등)를 기반으로 칼로리 소모량을 예측하는 회귀 모델을 구축하는 것입니다.

## 📌 전체 과정
1. 데이터 로드 및 탐색
2. 전처리 및 파생변수 생성
3. 모델링 및 검증 전략 설계
4. 모델 성능 비교 및 변수 선택
5. 하이퍼파라미터 튜닝
6. 앙상블 (Stacking)
7. 최종 예측 및 제출

> 당시에는 성능 개선을 위해 다양한 시도를 빠르게 반복하는 방식으로 진행했습니다.

In [ ]:
# 대회 당시 설치했던 패키지
# (여기서는 requirements.txt로 관리)

import os
import random
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, early_stopping
from catboost import CatBoostRegressor

import optuna
from optuna.samplers import TPESampler

## 재현성을 위한 랜덤 시드 고정

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

## 데이터 로드 및 기본 탐색

- ID 제거 후 feature / target 분리
- info / describe 기반 구조 확인

In [ ]:
train = pd.read_csv('train.csv')

train_x = train.drop(['ID', 'Calories_Burned'], axis=1)
train_y = train['Calories_Burned']

print(train_x.info())
print(train_x.describe())

## 데이터 전처리 및 파생변수 생성

### ✔ 수행 내용
- 단위 변환 (키, 몸무게, 체온)
- 범주형 인코딩
- 생리학 기반 변수 생성 (BMI, BMR 등)
- 운동 강도 관련 파생 변수 생성
- 비선형 변환 및 로그 변환

> 당시 성능 향상의 핵심은 **Feature Engineering** 이었습니다.

In [ ]:
# 전처리 함수
def preprocess_data(df):
    # 1. 기존 데이터 변환(단위 변환 & 범주형 매핑)
    df['Height_cm'] = (df['Height(Feet)'] * 30.48) + (df['Height(Remainder_Inches)'] * 2.54)
    df['Weight_kg'] = df['Weight(lb)'] * 0.4535
    df['Temp_C'] = (df['Body_Temperature(F)'] - 32) * 5 / 9
    df['Weight_Status'] = df['Weight_Status'].map({'Normal Weight': 0, 'Overweight': 1, 'Obese': 2})
    df['Gender'] = df['Gender'].map({'M': 0, 'F': 1})

    # 2. 파생변수 생성 (기존 사용자 변수 + 고도화 변수 통합)
    df['BMI'] = df['Weight_kg'] / ((df['Height_cm'] / 100) ** 2)

    # BMR 계산
    # 기초대사량 (BMR) - Mifflin-St Jeor 공식
    # 남성: 10*weight + 6.25*height - 5*age + 5
    # 여성: 10*weight + 6.25*height - 5*age - 161
    common_bmr = (10 * df['Weight_kg']) + (6.25 * df['Height_cm']) - (5 * df['Age'])
    df['BMR'] = np.where(df['Gender'] == 0, common_bmr + 5, common_bmr - 161)

    # 심박수 관련 기초 지표
    df['Max_HR'] = 208 - (0.7 * df['Age'])
    df['HR_Ratio'] = df['BPM'] / df['Max_HR']
    df['Temp_Deviation'] = df['Temp_C'] - 36.5

    # [사용자 변수 유지]
    df['Intensity_Factor'] = df['BPM'] * df['Exercise_Duration']
    df['Weight_Load'] = df['Weight_kg'] * df['Exercise_Duration']
    # High_Burn_Zone에 로그 변환 적용 (SHAP plateau 현상 해결)
    raw_burn = (df['HR_Ratio'] ** 3) * df['Exercise_Duration']
    df['High_Burn_Zone'] = np.log1p(raw_burn)

    df['Weight_BPM'] = df['Weight_kg'] * df['BPM']
    df['Weight_Cardio_Eff'] = (df['BPM'] * df['Weight_kg']) / (df['Age'] + 1)
    df['Effort_Per_Minute'] = (df['Max_HR'] - df['BPM']) / (df['Exercise_Duration'] + 1)
    df['Metabolic_Heat_Rate'] = (df['Temp_Deviation'] * df['Exercise_Duration']) / (df['BMR'] + 1)
    df['Intensity_Factor_sqrt'] = np.sqrt(df['Intensity_Factor'])

    # [추가된 고도화 변수]
    df['HRR_Duration'] = df['HR_Ratio'] * df['Exercise_Duration']
    df['Energy_Index'] = df['BMR'] * df['HR_Ratio'] * df['Exercise_Duration']
    df['High_Intensity_Burst'] = np.where(df['BPM'] > 110, (df['BPM'] - 110)**1.5 * df['Exercise_Duration'], 0)
    df['Heat_Stress_Pivot'] = np.maximum(0, df['Temp_C'] - 39.0) * df['Exercise_Duration']
    df['Weight_Intensity_Log'] = np.log1p(df['Weight_kg'] * df['HR_Ratio'] * df['Exercise_Duration'])
    df['Cardio_Efficiency'] = df['BMR'] / (df['BPM'] + 1)
    df['Intensity_Density'] = df['High_Burn_Zone'] / (df['Age'] + 1)
    df['Heat_Strain'] = df['Temp_Deviation'] * df['Exercise_Duration']
    df['Heat_Per_Weight'] = df['Heat_Strain'] / df['Weight_kg']
    df['BPM_Range'] = df['BPM'] - (df['Max_HR'] * 0.5)

    # 비선형 및 이상치 지표
    df['BPM_sq'] = df['BPM'] ** 2
    df['HRR_Duration_sqrt'] = np.sqrt(df['HRR_Duration'])
    df['BPM_Duration_Log'] = np.log1p(df['Intensity_Factor'])

    # 고강도 구간 오차 막는 변수
    df['HBZ_Duration_Ratio'] = df['High_Burn_Zone'] / (df['HRR_Duration'] + 1e-6) # 고강도 지속 효율 (고강도 구간에서 얼마나 효율적으로 태웠나)
    df['HBZ_BPM_Range'] = df['High_Burn_Zone'] * df['BPM_Range'] # 심박 변동과 고강도의 결합 (심장 부하량)
    df['Weight_Intensity_Factor'] = df['Weight_kg'] * df['Intensity_Factor'] # 체중 대비 강도 (신체 조건에 따른 강도 보정)
    df['HBZ_Squared'] = df['High_Burn_Zone'] ** 2 # 고강도 구간의 비선형 강조 (Dependence Plot의 꺾임 보정)

    # 신체 효율 지수 및 이상치 플래그
    df['Effort_Ratio'] = df['HR_Ratio'] * df['Exercise_Duration'] / df['Weight_kg']
    q99 = df['Effort_Ratio'].quantile(0.99)
    df['Is_Extreme'] = (df['Effort_Ratio'] > q99).astype(int)

    # 3. 불필요 컬럼 삭제
    drop_cols = ['Height(Feet)', 'Height(Remainder_Inches)', 'Weight(lb)', 'Body_Temperature(F)', 'Temp_C']
    df = df.drop(columns=drop_cols)

    return df

## 클러스터링 기반 Feature 추가

- Gaussian Mixture Model 사용
- Cluster를 One-Hot Encoding으로 변환

> 단순 feature 외에 데이터 구조 자체를 반영하려는 시도

In [ ]:
train_x = preprocess_data(train_x)

scaler = StandardScaler()
train_x_scaled = scaler.fit_transform(train_x)

gmm = GaussianMixture(n_components=4, random_state=42)
train_x['Cluster'] = gmm.fit_predict(train_x_scaled)

oh_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
oh_enc.fit(train_x[['Cluster']])

train_cluster_encoded = oh_enc.transform(train_x[['Cluster']])
cluster_cols = [f'Cluster_{i}' for i in range(train_cluster_encoded.shape[1])]
train_cluster_df = pd.DataFrame(train_cluster_encoded, columns=cluster_cols, index=train_x.index)

train_x = pd.concat([train_x.drop(columns=['Cluster']), train_cluster_df], axis=1)

## 평가 전략

### ✔ 핵심 아이디어
- "총 칼로리" 대신 "분당 칼로리"로 학습
- 이후 운동 시간 곱해서 복원

### ✔ 이유
- 타겟 분포 왜곡 문제 해결
- 모델 학습 안정성 확보

In [ ]:
# 타겟 변환 (분당 소모 칼로리)
y_data_rate = train_y / train_x['Exercise_Duration']

def evaluate_model_rate_cv(model, X, y_rate, original_y, duration_col, log_transform=False):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    metrics = {'RMSE': [], 'MAE': [], 'R2': []}

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train_rate = y_rate.iloc[train_idx]
        y_val_orig = original_y.iloc[val_idx] # 실제 칼로리 정답
        val_duration = X_val[duration_col]    # 복원용 시간

        if log_transform:
            y_train_rate = np.log1p(y_train_rate)

        model.fit(X_train, y_train_rate)
        preds_rate = model.predict(X_val)

        if log_transform:
            preds_rate = np.expm1(preds_rate)

        # 핵심: [예측된 분당 소모량 * 운동 시간] = 최종 예측 칼로리 복원
        final_preds = preds_rate * val_duration

        # 평가는 실제 칼로리 오차로 계산
        metrics['RMSE'].append(root_mean_squared_error(y_val_orig, final_preds))
        metrics['MAE'].append(mean_absolute_error(y_val_orig, final_preds))
        metrics['R2'].append(r2_score(y_val_orig, final_preds))

    return {k: np.mean(v) for k, v in metrics.items()}

## 로그 변환 효과 검증

- 타겟이 Right-Skewed → 로그 변환 실험
- 하나의 모델(XGB)로 검증 후 전체 모델에 적용 여부 결정

In [ ]:
# 실행(대표로 xgb 모델 하나)
xgb_baseline = XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=42)

print("Rate Target - Raw Scale")
raw_results = evaluate_model_rate_cv(
    xgb_baseline,
    train_x,
    y_data_rate,         # 위에서 생성한 분당 칼로리 데이터
    train_y,             # 원본 칼로리 데이터
    'Exercise_Duration',
    log_transform=False
)

print("Rate Target - Log Scale")
log_results = evaluate_model_rate_cv(
    xgb_baseline,
    train_x,
    y_data_rate,
    train_y,
    'Exercise_Duration',
    log_transform=True
)

# 결과 출력 테이블
results_df = pd.DataFrame([raw_results, log_results], index=['Rate_Raw', 'Rate_Log'])
print(results_df)
# 👉 Log 변환이 더 낮은 RMSE를 보여 최종적으로 Log 적용 결정

## 모델 성능 비교

- XGBoost
- LightGBM
- CatBoost

> 트리 기반 모델 3종 비교

In [ ]:
# 각 모델별 기본 체급 확인용
# 일반적으로 XGB와 LGBM의 경우, n_estimators 값을 300~1000 사이에서 성능이 안정되고,
# CatBoost는 1000 이상 줘야 제대로 성능 나오는 경우가 많기 때문에
# 이러한 점들을 고려하여 안정적인 세팅으로 XGB/LGBM은 300, CAT은 1000으로 설정.
use_log_rate = True # 로그 변환 결과에 따라서 True/False. Log의 결과가 조금 더 좋으므로 Log 사용.

models = {
    "CAT": CatBoostRegressor(iterations=1000, learning_rate=0.1, random_state=42, verbose=0),
    "XGB": XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=42),
    "LGBM": LGBMRegressor(n_estimators=300, learning_rate=0.1, random_state=42)
}

print(f"=== 기본 체급 확인 (Target: 분당 칼로리 / 모든 변수 사용 / Log: {use_log_rate}) ===")
print("-" * 75)

for name, model in models.items():
    # evaluate_model_rate_cv 함수를 사용하여 '복원된 RMSE'를 측정합니다.
    score = evaluate_model_rate_cv(
        model,
        train_x,          # 모든 변수 포함 데이터
        y_data_rate,      # 모델이 학습할 목표 (분당 칼로리)
        train_y,          # 원본 정답 (최종 평가용)
        'Exercise_Duration',
        log_transform=use_log_rate
    )

    print(f"{name:<4} CV RMSE (복원): {score['RMSE']:.6f} | MAE: {score['MAE']:.6f} | R2: {score['R2']:.6f}")
print("-" * 75)

## 변수 중요도 기반 Feature Selection

- 모델별 중요도 추출
- 불필요 변수 제거
- 모델별 최적 feature set 구성

In [ ]:
# 모델별 개별 변수 중요도 테이블 추출 (Rate 학습 기준)
# 1. 모델 정의
models = {
    "CAT": CatBoostRegressor(iterations=1000, learning_rate=0.1, random_state=42, verbose=0),
    "XGB": XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=42),
    "LGBM": LGBMRegressor(n_estimators=300, learning_rate=0.1, random_state=42)
}

# 2. 모델별 개별 테이블 생성 및 출력
for name, model in models.items():
    print(f"\n{'='*20} {name} Feature Importance (Target: Rate_Log) {'='*20}")

    # Log 변환 타겟을 학습(점수가 조금 더 좋았기 때문)
    model.fit(train_x, np.log1p(y_data_rate))

    if name == "CAT":
        imp = model.get_feature_importance()
    else:
        imp = model.feature_importances_

    # 개별 데이터프레임 생성
    df_imp = pd.DataFrame({
        "Feature": train_x.columns,
        "Importance": imp
    }).sort_values(by="Importance", ascending=False).reset_index(drop=True)

    # 상위 20개 출력
    display(df_imp.head(20))

    # 전역 변수로 저장
    globals()[f"{name.lower()}_imp_df"] = df_imp

In [ ]:
# 각 모델별 변수 제거 따로 해주기
# 전진선택법 사용 -> 제일 낮은 rmse를 구성하는 변수들 이외의 변수들을 전부 drop하는 방식으로 진행
# 따라서 아래 3개의 셀(전진 선택법)로 먼저 진행한 뒤, 이 셀 실행.

# 1. CAT 전용
cols_to_drop_cat = [
    'HBZ_Squared', 'Weight_BPM', 'High_Burn_Zone', 'Max_HR',
    'Cardio_Efficiency', 'HBZ_Duration_Ratio', 'Cluster_3',
    'Heat_Per_Weight', 'Intensity_Density', 'Temp_Deviation',
    'HRR_Duration', 'High_Intensity_Burst', 'Weight_Cardio_Eff',
    'Height_cm', 'BMI', 'Effort_Ratio', 'Metabolic_Heat_Rate',
    'Effort_Per_Minute', 'Heat_Strain', 'HRR_Duration_sqrt',
    'Intensity_Factor', 'Intensity_Factor_sqrt', 'Weight_Intensity_Log',
    'Weight_Intensity_Factor', 'Heat_Stress_Pivot', 'Weight_Load',
    'Weight_Status', 'Energy_Index', 'Is_Extreme',
    'Cluster_0', 'Cluster_1', 'Cluster_2',
]
train_x_cat = train_x.drop(columns=cols_to_drop_cat)

# 2. XGB 전용
cols_to_drop_xgb = [
    'High_Intensity_Burst', 'Cluster_3', 'BMR',
    'HBZ_Duration_Ratio', 'Effort_Ratio', 'Age',
    'Weight_Intensity_Log', 'BMI', 'Heat_Strain', 'BPM_sq',
    'High_Burn_Zone', 'Max_HR', 'Temp_Deviation',
    'BPM_Duration_Log', 'HRR_Duration', 'Metabolic_Heat_Rate',
    'HRR_Duration_sqrt', 'Intensity_Factor', 'Intensity_Factor_sqrt',
    'Heat_Stress_Pivot', 'Weight_Load', 'Weight_Status',
    'Energy_Index', 'Is_Extreme', 'Cluster_0', 'Cluster_2',
    'Intensity_Density', 'Heat_Per_Weight'
]
train_x_xgb = train_x.drop(columns=cols_to_drop_xgb)

# 3. LGBM 전용
cols_to_drop_lgbm = [
    'Age', 'HBZ_Duration_Ratio', 'Intensity_Density',
    'HBZ_BPM_Range', 'High_Burn_Zone', 'Effort_Ratio',
    'Effort_Per_Minute', 'Metabolic_Heat_Rate', 'Height_cm',
    'Heat_Per_Weight', 'BPM_sq', 'HBZ_Squared', 'HRR_Duration',
    'HRR_Duration_sqrt', 'Intensity_Factor', 'Intensity_Factor_sqrt',
    'BPM_Duration_Log', 'Weight_Intensity_Log', 'Weight_Intensity_Factor',
    'Heat_Strain', 'Temp_Deviation', 'Heat_Stress_Pivot',
    'Energy_Index', 'Is_Extreme', 'Cluster_0', 'Cluster_1', 'Cluster_3'
]
train_x_lgbm = train_x.drop(columns=cols_to_drop_lgbm)

# 4. 각 모델 변수 제거 후 CV RMSE 확인 (Rate 학습 & 복원 평가 방식)
print("="*80)
print(f"{'Model':<10} | {'CV RMSE (복원)':<15} | {'MAE':<10} | {'R2':<10}")
print("-"*80)

# 공통 파라미터 및 설정
target_rate = y_data_rate  # 분당 칼로리 타겟
duration_col = 'Exercise_Duration'

# --- CAT 확인 ---
score_cat = evaluate_model_rate_cv(
    CatBoostRegressor(iterations=1000, learning_rate=0.1, random_state=42, verbose=0),
    train_x_cat, target_rate, train_y, duration_col, log_transform=True
)
print(f"{'CAT':<10} | {score_cat['RMSE']:<15.6f} | {score_cat['MAE']:<10.6f} | {score_cat['R2']:<10.6f}")

# --- XGB 확인 ---
score_xgb = evaluate_model_rate_cv(
    XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=42, n_jobs=-1),
    train_x_xgb, target_rate, train_y, duration_col, log_transform=True
)
print(f"{'XGB':<10} | {score_xgb['RMSE']:<15.6f} | {score_xgb['MAE']:<10.6f} | {score_xgb['R2']:<10.6f}")

# --- LGBM 확인 ---
score_lgbm = evaluate_model_rate_cv(
    LGBMRegressor(n_estimators=300, learning_rate=0.1, random_state=42, verbose=-1),
    train_x_lgbm, target_rate, train_y, duration_col, log_transform=True
)
print(f"{'LGBM':<10} | {score_lgbm['RMSE']:<15.6f} | {score_lgbm['MAE']:<10.6f} | {score_lgbm['R2']:<10.6f}")

print("="*80)

## Forward Selection 실험

- 중요 변수부터 시작, 이후 변수를 하나씩 추가하며 RMSE가 개선되는 경우에만 유지하는 방식으로 선택
- (Greedy Forward Selection)

> 매우 수작업 기반 실험 방식

In [ ]:
# [실험 셀] CAT Forward Selection

# 1. 고정 변수 (중요도 상위 5개로 시작)
core_features = [
    'Exercise_Duration', 'BPM_Range', 'HR_Ratio', 'Gender',
    'HBZ_BPM_Range', 'BPM'
    # CAT 복원 RMSE: 3.282637 -> 5개 변수들로만 했을 때의 rmse 점수.
]

# 2. 후보 변수 (나머지 모든 변수. 하나씩 주석 풀면서 비교해서 더 낮은 rmse 결과 변수 생존)
candidate_features = [
    'BMR', # 0.955056
    'BPM_sq', # 0.893095
    'Weight_kg', # 0.769362
    # 'HBZ_Squared',
    # 'Weight_BPM',
    'Age', # 0.654143
    # 'High_Burn_Zone',
    # 'Max_HR',
    # 'Cardio_Efficiency',
    # 'HBZ_Duration_Ratio',
    # 'Cluster_3',
    # 'Heat_Per_Weight',
    # 'Intensity_Density',
    # 'Temp_Deviation',
    'BPM_Duration_Log', # 0.650164(최종)
    # 'HRR_Duration',
    # 'High_Intensity_Burst',
    # 'Weight_Cardio_Eff',
    # 'Height_cm',
    # 'BMI',
    # 'Effort_Ratio',
    # 'Metabolic_Heat_Rate',
    # 'Effort_Per_Minute',
    # 'Heat_Strain',
    # 'HRR_Duration_sqrt',
    # 'Intensity_Factor',
    # 'Intensity_Factor_sqrt',
    # 'Weight_Intensity_Log',
    # 'Weight_Intensity_Factor',
    # 'Heat_Stress_Pivot',
    # 'Weight_Load',
    # 'Weight_Status',
    # 'Energy_Index',
    # 'Is_Extreme',
    # 'Cluster_0',
    # 'Cluster_1',
    # 'Cluster_2',
]

# 3. 선택된 변수 조합 (core에 있는 변수 + candidate 주석 없는 변수)
selected_features = core_features + candidate_features

train_x_cat_test = train_x[selected_features]
print(f"현재 테스트 변수 개수: {len(selected_features)}")

# 4. 새로운 평가 로직 (0.784 기준. 모든 변수 포함 시 cat 모델의 기본 체급 rmse)
res_cat = evaluate_model_rate_cv(
    CatBoostRegressor(iterations=1000, learning_rate=0.1, random_state=42, verbose=0),
    train_x_cat_test,
    y_data_rate,
    train_y,
    'Exercise_Duration',
    log_transform=True # CAT 체급 확인 시 True가 결과가 좋았으므로 유지
)

print(f"CAT 복원 RMSE: {res_cat['RMSE']:.6f}")

In [ ]:
# [실험 셀] XGB Forward Selection

# 1. XGB 중요도 상위권
core_features = [
    'Exercise_Duration', 'HBZ_BPM_Range', 'BPM_Range',
    'HR_Ratio', 'Gender', 'BPM'
    # XGB 복원 RMSE: 3.336487
]

# 2. 후보 변수 (마찬가지로, 주석 하나씩 풀면서 더 낮은 rmse 값 도출하는 변수만 남기기)
candidate_features = [
    'Cluster_1', # 3.311872
    'Cardio_Efficiency', # 1.078393
    'Weight_BPM', # 0.984565
    'Weight_kg', # 0.944437
    # 'High_Intensity_Burst',
    'Weight_Cardio_Eff', # 0.941733
    # 'Cluster_3',
    # 'BMR',
    # 'HBZ_Duration_Ratio',
    # 'Effort_Ratio',
    'Height_cm', # 0.930497
    # 'Age',
    # 'Weight_Intensity_Log',
    # 'BMI',
    # 'Heat_Strain',
    # 'BPM_sq',
    'HBZ_Squared', # 0.927481
    # 'High_Burn_Zone',
    # 'Max_HR',
    # 'Temp_Deviation',
    # 'BPM_Duration_Log',
    # 'HRR_Duration',
    # 'Metabolic_Heat_Rate',
    'Effort_Per_Minute', # 0.924559
    # 'HRR_Duration_sqrt',
    # 'Intensity_Factor',
    # 'Intensity_Factor_sqrt',
    'Weight_Intensity_Factor', # 0.922435(최종)
    # 'Heat_Stress_Pivot',
    # 'Weight_Load',
    # 'Weight_Status',
    # 'Energy_Index',
    # 'Is_Extreme',
    # 'Cluster_0',
    # 'Cluster_2',
    # 'Intensity_Density',
    # 'Heat_Per_Weight'
]

# 3. 변수 조합
selected_features = core_features + candidate_features

train_x_xgb_test = train_x[selected_features]
print(f"현재 테스트 변수 개수: {len(selected_features)}")

# 4. 평가 로직 (XGB 전용)
res_xgb = evaluate_model_rate_cv(
    XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=42, n_jobs=-1),
    train_x_xgb_test,
    y_data_rate,
    train_y,
    'Exercise_Duration',
    log_transform=True
)

print(f"XGB 복원 RMSE: {res_xgb['RMSE']:.6f}")

In [ ]:
# [실험 셀] LGBM Forward Selection

# 1. LGBM 중요도 상위권
core_features = [
    'Exercise_Duration', 'Cardio_Efficiency', 'BPM', 'HR_Ratio',
    'BPM_Range', 'Weight_BPM'
    # LGBM 복원 RMSE: 2.305601
]

# 2. 후보 변수
candidate_features = [
    'Weight_kg', # 2.173934
    'Weight_Cardio_Eff', # 2.138114
    'BMI', # 1.932291
    'BMR', # 1.725062
    'Gender', # 0.995320
    'Max_HR', # 0.988964
    'Weight_Load', # 0.987102
    'High_Intensity_Burst', # 0.969312
    'Cluster_2', # 0.969051
    'Weight_Status', # 0.965543(최종)
    # 'Age',
    # 'HBZ_Duration_Ratio',
    # 'Intensity_Density',
    # 'HBZ_BPM_Range',
    # 'High_Burn_Zone',
    # 'Effort_Ratio',
    # 'Effort_Per_Minute',
    # 'Metabolic_Heat_Rate',
    # 'Height_cm',
    # 'Heat_Per_Weight',
    # 'BPM_sq',
    # 'HBZ_Squared',
    # 'HRR_Duration',
    # 'HRR_Duration_sqrt',
    # 'Intensity_Factor',
    # 'Intensity_Factor_sqrt',
    # 'BPM_Duration_Log',
    # 'Weight_Intensity_Log',
    # 'Weight_Intensity_Factor',
    # 'Heat_Strain',
    # 'Temp_Deviation',
    # 'Heat_Stress_Pivot',
    # 'Energy_Index',
    # 'Is_Extreme',
    # 'Cluster_0',
    # 'Cluster_1',
    # 'Cluster_3'
]

# 3. 변수 조합
selected_features = core_features + candidate_features

train_x_lgbm_test = train_x[selected_features]
print(f"현재 테스트 변수 개수: {len(selected_features)}")

# 4. 평가 로직 (LGBM 전용)
res_lgbm = evaluate_model_rate_cv(
    LGBMRegressor(n_estimators=300, learning_rate=0.1, random_state=42, verbose=-1),
    train_x_lgbm_test,
    y_data_rate,
    train_y,
    'Exercise_Duration',
    log_transform=True
)

print(f"LGBM 복원 RMSE: {res_lgbm['RMSE']:.6f}")

## 변수 제거 후 모델별 개별 중요도 확인 (Rate_Log 학습 기준)

In [ ]:
# 1. 모델 및 데이터 설정
model_configs = {
    "CAT": {
        "model": CatBoostRegressor(iterations=1000, learning_rate=0.1, random_state=42, verbose=0),
        "data": train_x_cat
    },
    "XGB": {
        "model": XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=42, n_jobs=-1),
        "data": train_x_xgb
    },
    "LGBM": {
        "model": LGBMRegressor(n_estimators=300, learning_rate=0.1, random_state=42, verbose=-1),
        "data": train_x_lgbm
    }
}

# 2. 중요도 계산 및 출력
for name, config in model_configs.items():
    print(f"\n{'='*20} {name} 정예 멤버 중요도 (Target: Rate_Log) {'='*20}")

    current_model = config["model"]
    current_x = config["data"]

    # 원본 train_y가 아닌, y_data_rate의 로그값 학습
    current_model.fit(current_x, np.log1p(y_data_rate))

    # 모델별 중요도 추출 방식 설정
    if name == "CAT":
        imp = current_model.get_feature_importance()
    else:
        imp = current_model.feature_importances_

    # 데이터프레임 생성
    df_imp = pd.DataFrame({
        "Feature": current_x.columns,
        "Importance": imp
    }).sort_values(by="Importance", ascending=False).reset_index(drop=True)

    # 결과 출력
    display(df_imp)

    # 개별 변수로 저장
    globals()[f"{name.lower()}_reduced_imp_df"] = df_imp

print("\n모든 모델의 최종 선택된 변수 중요도 확인 완료")

## 하이퍼파라미터 튜닝 (Optuna)

- CatBoost / XGBoost / LightGBM 각각 튜닝
- RMSE 기준 최적화

In [ ]:
# 공통 설정
sampler = TPESampler(seed=42)

# 1. CatBoost용 목적 함수
def objective_cat(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 2000, 5000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 10.0),
        "random_strength": trial.suggest_float("random_strength", 1.0, 8.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 128, 255),
        "random_state": 42,
        "verbose": 0,
        "allow_writing_files": False
    }
    model = CatBoostRegressor(**params)
    # rate_cv 함수 사용 (y_data_rate와 train_y를 모두 전달)
    score = evaluate_model_rate_cv(model, train_x_cat, y_data_rate, train_y, 'Exercise_Duration', log_transform=True)
    return score['RMSE']

# 2. LightGBM용 목적 함수
def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 1000, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.07),
        "num_leaves": trial.suggest_int("num_leaves", 31, 128),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 30),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 10.0, log=True),
        "random_state": 42, "n_jobs": -1, "verbose": -1
    }
    model = LGBMRegressor(**params)
    score = evaluate_model_rate_cv(model, train_x_lgbm, y_data_rate, train_y, 'Exercise_Duration', log_transform=True)
    return score['RMSE']

# 3. XGBoost용 목적 함수
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 1000, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.07),
        "max_depth": trial.suggest_int("max_depth", 4, 9),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 10.0, log=True),
        "random_state": 42, "n_jobs": -1, "verbosity": 0
    }
    model = XGBRegressor(**params)
    score = evaluate_model_rate_cv(model, train_x_xgb, y_data_rate, train_y, 'Exercise_Duration', log_transform=True)
    return score['RMSE']

# --- 최적화 실행 ---
print("Tuning CatBoost...")
study_cat = optuna.create_study(direction="minimize", sampler=sampler)
study_cat.optimize(objective_cat, n_trials=100) # 성능이 좋으므로 더 많이

print("\nTuning LightGBM...")
study_lgbm = optuna.create_study(direction="minimize", sampler=sampler)
study_lgbm.optimize(objective_lgbm, n_trials=40)

print("\nTuning XGBoost...")
study_xgb = optuna.create_study(direction="minimize", sampler=sampler)
study_xgb.optimize(objective_xgb, n_trials=40)

# --- 결과 요약 및 보고 ---
best_models_info = {
    "CAT": (study_cat, train_x_cat, CatBoostRegressor),
    "LGBM": (study_lgbm, train_x_lgbm, LGBMRegressor),
    "XGB": (study_xgb, train_x_xgb, XGBRegressor)
}

print("\n" + "=" * 70)
print(f"{'Model':<8} | {'Best RMSE':<15} | {'Best Parameters Found'}")
print("-" * 70)

for name, (study, x_data, model_cls) in best_models_info.items():
    print(f"{name:<8} | {study.best_value:<15.6f} | {study.best_params}")

print("=" * 70)

# --- 튜닝 결과 정리 및 최종 가중치 산정을 위한 루프 ---
final_models_summary = []

print("\n" + "=" * 80)
print(f"{'Model':<8} | {'Final RMSE':<12} | {'Final MAE':<12} | {'Final R2':<12}")
print("-" * 80)

for name, (study, x_data, model_cls) in best_models_info.items():
    # 1. 각 모델별 최적의 파라미터 추출
    best_params = study.best_params

    # 2. 모델 객체 생성
    if name == "CAT":
        # verbose 조절을 위함
        final_model = model_cls(**best_params, random_state=42, verbose=0)
    else:
        final_model = model_cls(**best_params, random_state=42)

    # 3. Rate 기반 복원 RMSE 재확인 (신뢰도 검증)
    metrics = evaluate_model_rate_cv(
        final_model, x_data, y_data_rate, train_y, 'Exercise_Duration', log_transform=True
    )

    print(f"{name:<8} | {metrics['RMSE']:<12.6f} | {metrics['MAE']:<12.6f} | {metrics['R2']:<12.6f}")

    # 4. 나중에 앙상블 때 써먹기 위해 파라미터와 점수를 리스트에 보관
    final_models_summary.append({
        'name': name,
        'params': best_params,
        'rmse': metrics['RMSE'],
        'model_class': model_cls,
        'train_x': x_data
    })

print("=" * 80)

## 앙상블 (Stacking)

- Base: CAT / XGB / LGBM
- Meta: Ridge

> 단일 모델보다 성능 향상 확인

In [ ]:
# 0. 준비 및 설정
kf = KFold(n_splits=5, shuffle=True, random_state=42)
n_samples = len(train_y)

# 타겟 설정 (분당 칼로리 비율의 로그값)
y_rate_log = np.log1p(y_data_rate)
duration_train = train_x['Exercise_Duration']

# 모델 저장용
models_cat, models_lgbm, models_xgb = [], [], []

# Out-of-Fold 예측값 (원본 칼로리 복원본 저장용)
oof_cat = np.zeros(n_samples)
oof_lgbm = np.zeros(n_samples)
oof_xgb = np.zeros(n_samples)

# 1. K-Fold 교차 검증 및 OOF 생성
for fold, (train_idx, val_idx) in enumerate(kf.split(train_y)):
    print(f"\n# Fold {fold+1} Training (Rate-Based)...")

    # 데이터 분할
    y_tr_rate, y_val_rate = y_rate_log.iloc[train_idx], y_rate_log.iloc[val_idx]
    val_duration = duration_train.iloc[val_idx]

    # --- CAT ---
    X_tr_cat, X_val_cat = train_x_cat.iloc[train_idx], train_x_cat.iloc[val_idx]
    cat_params = study_cat.best_params.copy()
    cat = CatBoostRegressor(**cat_params, random_state=42, verbose=0, early_stopping_rounds=300)
    cat.fit(X_tr_cat, y_tr_rate, eval_set=(X_val_cat, y_val_rate), use_best_model=True)
    oof_cat[val_idx] = np.expm1(cat.predict(X_val_cat)) * val_duration
    models_cat.append(cat)

    # --- LGBM ---
    X_tr_lgbm, X_val_lgbm = train_x_lgbm.iloc[train_idx], train_x_lgbm.iloc[val_idx]
    lgbm_params = study_lgbm.best_params.copy()
    lgbm = LGBMRegressor(**lgbm_params, random_state=42, verbose=-1)
    lgbm.fit(X_tr_lgbm, y_tr_rate, eval_set=[(X_val_lgbm, y_val_rate)], callbacks=[early_stopping(stopping_rounds=300)])
    oof_lgbm[val_idx] = np.expm1(lgbm.predict(X_val_lgbm)) * val_duration
    models_lgbm.append(lgbm)

    # --- XGB ---
    X_tr_xgb, X_val_xgb = train_x_xgb.iloc[train_idx], train_x_xgb.iloc[val_idx]
    xgb_params = study_xgb.best_params.copy()
    xgb = XGBRegressor(**xgb_params, random_state=42, verbosity=0, early_stopping_rounds=300)
    xgb.fit(X_tr_xgb, y_tr_rate, eval_set=[(X_val_xgb, y_val_rate)], verbose=False)
    oof_xgb[val_idx] = np.expm1(xgb.predict(X_val_xgb)) * val_duration
    models_xgb.append(xgb)

    fold_rmse = root_mean_squared_error(train_y.iloc[val_idx], oof_cat[val_idx])
    print(f"Fold {fold+1} Done. [CAT 복원 RMSE: {fold_rmse:.4f}]")

# 2. 메타 모델 분석 (Ridge Stacking)
stack_train = np.column_stack((oof_cat, oof_lgbm, oof_xgb))
meta_model = Ridge(alpha=1.0)
meta_model.fit(stack_train, train_y)
stack_pred = meta_model.predict(stack_train)

print("\n" + "="*50)
print(f"1. Ridge Stacking RMSE: {root_mean_squared_error(train_y, stack_pred):.6f}")
print(f"2. CAT Single RMSE: {root_mean_squared_error(train_y, oof_cat):.6f}")

# 수동 블렌딩
manual_blend = (oof_cat * 0.8) + (oof_xgb * 0.1) + (oof_lgbm * 0.1)
print(f"3. Manual Blend (CAT 80%) RMSE: {root_mean_squared_error(train_y, manual_blend):.6f}")

# 3가지 결과 중 가장 성적이 좋은 방식 사용
# 여기서는 1번이 가장 좋았음

## SHAP 기반 해석

- 모델 중요 변수 확인
- 고강도 구간(약 230kcal 이상)에서 예측이 실제보다 낮게 나오는 경향 확인
- 이를 기반으로 후처리 보정 로직 설계

In [ ]:
# SHAP & 오차 분석 셀

# 1. 대표 모델 선정 (가장 좋은 성능을 가진 CAT의 첫 번째 폴드)
best_cat_model = models_cat[0]
current_x = train_x_cat # 분석할 데이터셋

# SHAP Explainer 설정
explainer = shap.TreeExplainer(best_cat_model)
shap_values = explainer.shap_values(current_x)

# 2. Summary Plot (전체적인 변수 영향력 확인)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, current_x)

# 3. 모델이 가장 중요하게 생각하는 변수 자동 추출
top_feature = current_x.columns[np.abs(shap_values).mean(0).argmax()]
print(f"모델이 가장 중요하게 판단한 변수: {top_feature}")

# 4. 오차 분석 (Residual Analysis)
# 실제 칼로리 - OOF 복원 예측값
errors = train_y - oof_cat

plt.figure(figsize=(15, 5))

# 가장 중요한 변수와 오차의 관계
plt.subplot(1, 2, 1)
plt.scatter(current_x[top_feature], errors, alpha=0.3, color='royalblue')
plt.axhline(0, color='red', linestyle='--')
plt.title(f'Error by {top_feature}')
plt.xlabel(top_feature)
plt.ylabel('Error (Actual - Pred)')

# 예측값 크기에 따른 오차 분포
plt.subplot(1, 2, 2)
plt.scatter(oof_cat, errors, alpha=0.3, color='forestgreen')
plt.axhline(0, color='red', linestyle='--')
plt.title('Error by Prediction Scale')
plt.xlabel('Predicted Calories')
plt.ylabel('Error')

plt.tight_layout()
plt.show()

## 최종 예측 및 제출

- Stacking 결과 사용
- 고강도 구간 보정 적용
- 여러 블렌딩 전략 실험

## 📌 결과

- Public Score: 0.48대 달성
- 핵심 성능 개선 요소:
  - Feature Engineering
  - Rate 기반 타겟 변환
  - Stacking + SHAP 기반 보정

In [ ]:
# [최종] Ridge Stacking & SHAP 기반 고강도 보정 통합 예측 셀

# 1. 데이터 불러오기 및 기본 전처리
test = pd.read_csv('test.csv')
submission = pd.read_csv('sample_submission.csv')

test_x = test.drop(['ID'], axis=1)
test_x = preprocess_data(test_x) # 사전에 정의된 전처리 함수

# 1-1. GMM Cluster 및 Scaler 적용
x_test_scaled = scaler.transform(test_x)
test_x['Cluster'] = gmm.predict(x_test_scaled)

# 1-2. Cluster 원-핫 인코딩 적용
test_cluster_encoded = oh_enc.transform(test_x[['Cluster']])
cluster_cols = [f'Cluster_{i}' for i in range(test_cluster_encoded.shape[1])]
test_cluster_df = pd.DataFrame(test_cluster_encoded, columns=cluster_cols, index=test_x.index)
test_x = pd.concat([test_x.drop(columns=['Cluster']), test_cluster_df], axis=1)

# 2. 모델별 전용 테스트 데이터셋 생성
test_x_cat = test_x.drop(columns=cols_to_drop_cat)
test_x_lgbm = test_x.drop(columns=cols_to_drop_lgbm)
test_x_xgb = test_x.drop(columns=cols_to_drop_xgb)

# 3. Base 모델 예측 (중요: Rate 기반 예측 후 Duration 곱하기)
duration_test = test_x['Exercise_Duration']

# 원본 복구: expm1(Rate_Log_Pred) * Exercise_Duration 로직 적용
test_cat_pred = np.mean([np.expm1(m.predict(test_x_cat)) for m in models_cat], axis=0) * duration_test
test_lgbm_pred = np.mean([np.expm1(m.predict(test_x_lgbm)) for m in models_lgbm], axis=0) * duration_test
test_xgb_pred = np.mean([np.expm1(m.predict(test_x_xgb)) for m in models_xgb], axis=0) * duration_test

# 4. Meta 모델 학습용 feature 생성 (Stacking)
stack_test = np.column_stack((test_cat_pred, test_lgbm_pred, test_xgb_pred))

# 5. 최종 예측 수행
# 5-1. Ridge 모델 최종 예측
final_ridge_preds = meta_model.predict(stack_test)
final_ridge_preds = np.maximum(0, final_ridge_preds) # 음수 방지

# 5-2. SHAP 기반 고강도 구간 후처리 보정
# SHAP 분석 결과 230kcal 이상 고강도 구간에서 Under-predict 경향을 확인했으므로 약 0.7% 상향 보정
final_refined_preds = np.where(final_ridge_preds > 230, final_ridge_preds * 1.007, final_ridge_preds)

# 6. Submission 파일 생성
submission['Calories_Burned'] = final_refined_preds
submission.to_csv('./submit_final_ridge_refined.csv', index=False)

# 7. 검증 및 상태 출력
print("="*50)
print(f"Refined Ridge Stacking Saved: submit_final_ridge_refined.csv (SHAP Optimized)")
print("="*50)
print(f"Final Average Calories (Refined): {final_refined_preds.mean():.4f}")
print(f"Max Calories Predicted: {final_refined_preds.max():.4f}")

In [ ]:
# 위 셀의 결과를 더 깎아보고자, 가중치를 조정하는 방식으로 변경
final_final_blend = (final_ridge_preds * 0.7) + (test_cat_pred * 0.3)

# 고강도 보정을 조금 더 세밀하게 (1.007 대신 1.005 정도)
final_final_blend = np.where(final_final_blend > 230, final_final_blend * 1.005, final_final_blend)

submission['Calories_Burned'] = final_final_blend
submission.to_csv('last_dance_blend.csv', index=False)

print("최종 블렌딩 파일 'last_dance_blend.csv' 저장 완료!")
print(f"평균 칼로리: {final_final_blend.mean():.4f}")

# 제출 결과: 0.49대 진입 성공(데이콘 결과)

In [ ]:
# 마지막 시도(0.4 초반대를 가기 위한 시도)
# 1. 가중치 미세 조정 (Ridge 60%, CAT 40% - 성능이 좋았던 CAT의 비중 높이기)
final_push_blend = (final_ridge_preds * 0.6) + (test_cat_pred * 0.4)

# 2. 3단계 초정밀 계단식 후처리
def apply_staged_correction(val):
    if val > 275:
        return val * 1.012  # 극고강도
    elif val > 250:
        return val * 1.007  # 고강도
    elif val > 225:
        return val * 1.004  # 중고강도
    return val

# 벡터화된 함수 적용
final_push_refined = np.array([apply_staged_correction(x) for x in final_push_blend])

# 3. 데이터 범위 안전장치 (음수 방지 및 물리적 최대치 보정)
final_push_refined = np.maximum(0, final_push_refined)

# 4. 제출 파일 생성
submission['Calories_Burned'] = final_push_refined
file_name = 'submit_final_11.csv'
submission.to_csv(file_name, index=False)

print(f"="*50)
print(f"제출 파일 생성 완료: {file_name}")
print(f"최종 예측 평균: {final_push_refined.mean():.4f}")
print(f"최종 예측 최대값: {final_push_refined.max():.4f}")
print("="*50)

# 최종 결과: 0.48대 진입(데이콘 결과)